### Loading Libraries

In [1]:
import numpy as np
import pandas as pd
import glob
import plotly.express as px
import folium

import os
from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError
from zipfile import ZipFile

In [11]:
output_dir = "../data"

In [12]:
citibike_df = pd.read_csv(f"{output_dir}/JC/JC2025.csv",
                         parse_dates=['started_at','ended_at'])

In [13]:
citibike_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1002704 entries, 0 to 1002703
Data columns (total 13 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   ride_id             1002704 non-null  str           
 1   rideable_type       1002704 non-null  str           
 2   started_at          1002704 non-null  datetime64[us]
 3   ended_at            1002704 non-null  datetime64[us]
 4   start_station_name  1002701 non-null  str           
 5   start_station_id    1002701 non-null  str           
 6   end_station_name    999469 non-null   str           
 7   end_station_id      998307 non-null   str           
 8   start_lat           1002702 non-null  float64       
 9   start_lng           1002702 non-null  float64       
 10  end_lat             999260 non-null   float64       
 11  end_lng             999260 non-null   float64       
 12  member_casual       1002704 non-null  str           
dtypes: datetime64[us](2), f

In [14]:
citibike_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,880A0159BA5275FB,electric_bike,2025-01-16 17:50:49.136,2025-01-16 17:57:00.710,Hilltop,JC019,Pershing Field,JC024,40.731169,-74.057574,40.742677,-74.051789,member
1,1A5E1E274B2AF0AD,electric_bike,2025-01-31 06:10:41.818,2025-01-31 06:22:09.499,Hilltop,JC019,Jackson Square,JC063,40.731169,-74.057574,40.711130,-74.078900,member
2,EA9928D3C05B8377,classic_bike,2025-01-09 16:42:50.213,2025-01-09 17:04:12.870,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member
3,3C42C367750B9292,electric_bike,2025-01-21 16:14:14.398,2025-01-21 16:37:10.458,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member
4,94D3B0265A7BDE1F,classic_bike,2025-01-30 16:38:18.840,2025-01-30 17:04:08.166,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,40.735938,-74.030305,member


In [15]:
citibike_df['ride_duration_min'] = (citibike_df["ended_at"] - citibike_df["started_at"]).dt.total_seconds()/60


In [17]:
citibike_df.shape

(1002704, 14)

In [18]:
citibike_df.columns

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual', 'ride_duration_min'],
      dtype='str')

### Drop Missing Values

In [20]:
citibike_df = citibike_df.dropna(
    subset=['ride_id', 'started_at', 'ended_at','start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
        ]
)

In [21]:
missing_values = citibike_df.isnull().sum().reset_index()

missing_values.columns = ['column_name', 'missing_count']

missing_values["missing_percentage"] = (missing_values["missing_count"] / len(citibike_df)) * 100

missing_values.sort_values(by="missing_percentage", ascending=False, inplace=True)

missing_values

,column_name,missing_count,missing_percentage
0,ride_id,0,0.0
1,rideable_type,0,0.0
2,started_at,0,0.0
3,ended_at,0,0.0
4,start_station_name,0,0.0
5,start_station_id,0,0.0
6,end_station_name,0,0.0
7,end_station_id,0,0.0
8,start_lat,0,0.0
9,start_lng,0,0.0


In [26]:
from utils import assign_season

In [27]:
citibike_df['date'] = citibike_df['started_at'].dt.date
citibike_df['month'] = citibike_df['started_at'].dt.to_period('M').astype(str)
citibike_df['month_name'] = citibike_df['started_at'].dt.month_name()
citibike_df['month_number'] = citibike_df['started_at'].dt.month
citibike_df['day_of_week'] = citibike_df['started_at'].dt.day_name()
citibike_df['hour'] = citibike_df['started_at'].dt.hour
citibike_df['season'] = citibike_df['month_number'].apply(assign_season)

In [28]:
citibike_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,...,end_lng,member_casual,ride_duration_min,date,month,month_name,month_number,day_of_week,hour,season
0,880A0159BA5275FB,electric_bike,2025-01-16 17:50:49.136,2025-01-16 17:57:00.710,Hilltop,JC019,Pershing Field,JC024,40.731169,-74.057574,...,-74.051789,member,6.192900,2025-01-16,2025-01,January,1,Thursday,17,Winter
1,1A5E1E274B2AF0AD,electric_bike,2025-01-31 06:10:41.818,2025-01-31 06:22:09.499,Hilltop,JC019,Jackson Square,JC063,40.731169,-74.057574,...,-74.078900,member,11.461350,2025-01-31,2025-01,January,1,Friday,6,Winter
2,EA9928D3C05B8377,classic_bike,2025-01-09 16:42:50.213,2025-01-09 17:04:12.870,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,...,-74.030305,member,21.377617,2025-01-09,2025-01,January,1,Thursday,16,Winter
3,3C42C367750B9292,electric_bike,2025-01-21 16:14:14.398,2025-01-21 16:37:10.458,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,...,-74.030305,member,22.934333,2025-01-21,2025-01,January,1,Tuesday,16,Winter
4,94D3B0265A7BDE1F,classic_bike,2025-01-30 16:38:18.840,2025-01-30 17:04:08.166,Hilltop,JC019,Hoboken Terminal - Hudson St & Hudson Pl,HB101,40.731169,-74.057574,...,-74.030305,member,25.822100,2025-01-30,2025-01,January,1,Thursday,16,Winter


### Storing Enriched Data

In [29]:
citibike_df.to_csv(f'{output_dir}/JC/JC2025_Enriched.csv',index=False)
